In [ ]:
import os
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "reproduce.py").is_file())
os.chdir(ROOT)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

PROJECT_PATH = Path("data/genomics")

In [ ]:
meta = pd.read_csv(PROJECT_PATH / "reference/metadata_complete.csv")
meta['population'] = meta['Strain'] + '_' + meta['Culture'].astype(str).str.zfill(2)
meta

In [ ]:
# loops through rows of meta, finds the tsv for each, and adds the information about that tsv file's day, culture and strain

dfs = []

for i, row in meta.iterrows():
    df = pd.read_csv(PROJECT_PATH / f"out/{row['FolderDate']}/{row['source_file']}/output/output.gd.tsv", sep='\t')
    df["Strain"] = row['Strain']
    df["Culture"] = row['Culture']
    df["Day"] = int(row['Day'])
    dfs.append(df)
df = pd.concat(dfs)


In [ ]:
select_lineage = "PL"
pops = meta.query(f'Strain=="{select_lineage}"')['population'].unique()
#timepoints = sorted(meta.query(f'Strain=="{select_lineage}"')['Day'].unique())
timepoints = [0,16]
print(pops)
print(timepoints)

In [ ]:
meta.query(f'population=="{select_lineage}_01"')

In [ ]:
def traceAlleleFreq(pop, min_freq=0.33):

    D = []
    sorted_meta = meta.query(f'population=="{pop}"').sort_values(by='Day')
    for i, row in sorted_meta.iterrows():
        df = pd.read_csv(PROJECT_PATH / f"out/{row['FolderDate']}/{row['source_file']}/output/output.gd.tsv", sep='\t')
        D.append(df)

    # Remove mutations detected in the wild-type background
    # 1. Select background mutations with strong signal
    wt_background=D[0].loc[D[0].frequency>0.01, 'position']
    #print(f"WT background mutations: {wt_background.values}")
    D_=[]

    #print(f"# of mutations\t# of (freq>{min_freq}):")
    for i in range(len(D)):
        # 2. Filter out mutations by position on the chromosome
        df=D[i][~D[i]['position'].isin(wt_background)]
        D_.append(df)
        #print(f"{df.shape[0]}\t{sum(df['frequency']>min_freq)}")
    
    tracked_positions=[]
    for i in range(len(D_)):
        d=D_[i]
        tracked_positions.extend(list(d.loc[ (d['frequency']>min_freq), 'position' ])) #& ~d['mutation_category'].isin(['mobile_element_insertion','large_deletion']), 'position' ] ))
    tracked_positions=set(tracked_positions)
    print(f"Population {pop} # of tracked mutations: {len(tracked_positions)}")

    ## 3. Trace the frequency of those mutations
    T=[]
    for pos in tracked_positions:

        freq=[]
        for i in range(len(D_)):
            d = D_[i]
            ind = d['position']==pos
            
            if sum(ind)<1:
                freq.append(0)
            else:
                freq.append( (d.loc[ind , 'frequency'].values)[0] )
                gene_name = d.loc[ind, 'gene_name'].values[0]
                gene_product = d.loc[ind, 'gene_product'].values[0]
                aa_ref_seq = d.loc[ind, 'aa_ref_seq'].values[0]
                aa_new_seq = d.loc[ind, 'aa_new_seq'].values[0]
                aa_ref_seq = d.loc[ind, 'aa_ref_seq'].values[0]
                new_seq = d.loc[ind, 'new_seq'].values[0]
                codon_ref_seq = d.loc[ind, 'codon_ref_seq'].values[0]
                aa_pos = d.loc[ind, 'aa_position'].values[0]
                gene_pos = d.loc[ind, 'gene_position'].values[0]
                mut_cat = d.loc[ind, 'mutation_category'].values[0]

        T.append({'position': pos, 'freq': np.round(freq,2), 
                'gene_name':gene_name, 'gene_product':gene_product,
                'aa_ref_seq':aa_ref_seq, 'aa_new_seq':aa_new_seq,
                'aa_pos': aa_pos, 'gene_pos':gene_pos, 'mut_cat': mut_cat,
                'new_seq': new_seq, 'codon_ref_seq': codon_ref_seq})

    T=pd.DataFrame(T)
    T.fillna({'aa_ref_seq': '',
              'aa_new_seq': '',
              'aa_pos': ''},inplace=True)
    
    nsi = T['aa_pos']==''
    T.loc[nsi,'label'] = T.loc[nsi,'gene_name'] + ' ' + T.loc[nsi, 'mut_cat']
    T.loc[~nsi, 'label'] = T.loc[~nsi,'gene_name'] + ' ' + T.loc[~nsi,'aa_ref_seq'] + T.loc[~nsi, 'aa_pos'].astype(str).str.rstrip('.0') + T.loc[~nsi,'aa_new_seq']
        # T.loc[nsi,'gene_pos'] + ' '
    T.sort_values(by=['mut_cat','gene_name'],ascending=False,inplace=True)
    T.reset_index(inplace=True,drop=True)
    
    return T

In [ ]:
af=[]
for pop in pops:
    af.append(traceAlleleFreq(pop, min_freq=0.1))
    af[-1].to_csv(f"data/genomics/data/processed/traced_alleles/{select_lineage}/{pop}.csv", index=False)

In [ ]:
pd.set_option("display.max_rows", 100)
af[3].head(100)

In [ ]:
for df in af:
    df['last_freq'] = df['freq'].apply(lambda x: x[-1])

In [ ]:
all = []
for ix, df in enumerate(af):
    dff = df.copy()
    dff['Pop'] = ix+1 
    all.append(dff)
combined_df=pd.concat(all)


combined_df['last_freq'] = combined_df['freq'].apply(lambda x: x[-1])
combined_df

In [ ]:
def get_mutation_group_and_labels(group):
    # """
    # Returns select mutations and alternate labels based on the specified group.
    
    # Parameters:
    #     group (str): The group to retrieve mutations for. Options are 'prs_phoQ' or 'other'.
    
    # Returns:
    #     tuple: A tuple containing a list of selected mutations and a dictionary of alternate labels.
    # """
    # Select mutations based on group
    if group == 'gyrA_rpoB_icd':
        select_mutations = [
            'gyrA G81D', 'gyrA S83W',  'gyrA D87Y', 'gyrA S83L',  'gyrA A119E',
            'rpoB H1237L', 'rpoB D1064G', 'rpoB E562D', 'rpoB M129R', 'rpoB small_indel', 'rpoB D1064N', 
            'icd H366H' 
        ]
        alt_labels = {
            'rpoB small_indel': 'rpoB indel',
        }
        manual_label_order = [
            'gyrA G81D', 'gyrA S83L',  'gyrA S83W',  'gyrA A119E', 'gyrA D87Y',
            'rpoB H1237L',  'rpoB small_indel','rpoB D1064N',  'rpoB D1064G', 'rpoB E562D', 'rpoB M129R', 
            'icd H366H' 
]
    elif group == 'other':
        select_mutations = [
            'rplJ P89S', 'trkH L185Q', 'feaB D41E',
            'appY/ompT small_indel', 'glvC small_indel',
            'arcB G258C', 'paaG S241C', 'tas small_indel',
            'pssL small_indel', 'sucA V141A',
            'arcA small_indel',  'ygfB P184Q',
            'gltP/yjcO snp_intergenic', 'arcA L65F', 'puuA D431D', 'arpA D661A',
            'fimB/fimE mobile_element_insertion', 'ftp/ompC snp_intergenic',
            'yhaC/rnpB snp_intergenic', 'yhiI V38V', 'recF I355T',
            'selB small_indel', 'elaD C252C', 'hlyE E42D',
            'yffN/yffO snp_intergenic', 'pgaC E86G'
        ]
        alt_labels = {
            'appY/ompT small_indel': 'appY/ompT indel',
            'glvC small_indel': 'glvC indel',
            'tas small_indel': 'tas indel',
            'pssL small_indel': 'pssL indel',
            'arcA small_indel': 'arcA indel',
            'gltP/yjcO snp_intergenic': 'gltP/yjcO snp intergenic',
            'fimB/fimE mobile_element_insertion': 'fimB/fimE IS5 insertion',
            'ftp/ompC snp_intergenic': 'ftp/ompC snp intergenic',
            'yhaC/rnpB snp_intergenic': 'yhaC/rnpB snp intergenic',
            'selB small_indel': 'selB indel',
            'yffN/yffO snp_intergenic': 'yffN/yffO snp intergenic',
        }
        manual_label_order = [
            'rplJ P89S', 'trkH L185Q', 'feaB D41E',
            'appY/ompT small_indel', 'glvC small_indel',
            'arcB G258C', 'paaG S241C', 'tas small_indel',
            'pssL small_indel', 'sucA V141A',
            'arcA small_indel',  'ygfB P184Q',
            'gltP/yjcO snp_intergenic', 'arcA L65F', 'puuA D431D', 'arpA D661A',
            'fimB/fimE mobile_element_insertion', 'ftp/ompC snp_intergenic',
            'yhaC/rnpB snp_intergenic', 'yhiI V38V', 'recF I355T',
            'selB small_indel', 'elaD C252C', 'hlyE E42D',
            'yffN/yffO snp_intergenic', 'pgaC E86G'
]
    else:
        raise ValueError("Invalid group. Options are 'prs_phoQ' or 'other'.")

    return select_mutations, alt_labels, manual_label_order


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.colors as mcolors
plt.rcParams["font.family"] = "Nimbus Roman"

manual_label_order = [
    'gyrA G81D', 'gyrA S83L', 'gyrA S83W', 'gyrA A119E', 'gyrA D87Y',
    'rpoB H1237L', 'rpoB indel', 'rpoB D1064N', 'rpoB D1064G',
    'rpoB E562D', 'rpoB M129R', 'icd H366H'
]

# Assuming `af` is a list of DataFrames with the following structure
# Example structure for af: [{'label': '...', 'gene_name': '...', 'freq': [..]}]
# af = [...]

standard_map = plt.cm.get_cmap('Reds')

# Create a new colormap that interpolates between white and the 'Greens' colormap
new_colors = standard_map(np.linspace(0, 1, 256))
new_colors[0] = np.array([1, 1, 1, 1])  # Replace the first color (for 0 values) with white
custom_map = mcolors.ListedColormap(new_colors)

# Define the gene names to include (set to None or empty list for no filtering)
designated_gene_names = ['gyrA', 'rpoB']  # Only gyrA as the designated gene name

# Extract unique labels and their counts from the data
all_labels = np.concatenate([df['label'].unique() for df in af])
unique_labels, counts = np.unique(all_labels, return_counts=True)

# Create a DataFrame with labels and counts
label_counts = pd.DataFrame({'label': unique_labels, 'count': counts})

# Map gene_name and the last freq value to labels
gene_name_map = {row['label']: row['gene_name'] for df in af for _, row in df.iterrows()}
freq_map = {row['label']: row['freq'][-1] for df in af for _, row in df.iterrows()}

label_counts['gene_name'] = label_counts['label'].map(gene_name_map)
label_counts['freq'] = label_counts['label'].map(freq_map)

# Define a mapping of old labels to new alternate labels
label_mapping = {
    'rpoB small_indel': 'rpoB indel',
    'gyrA G81D': 'GyrA G81D',
    'gyrA S83L': 'GyrA S83L',
    'gyrA S83W': 'GyrA S83W',
    'gyrA A119E': 'GyrA A119E',
    # Add other mappings if needed
}

# Apply the label mapping to the label_counts DataFrame
label_counts['label'] = label_counts['label'].replace(label_mapping)
combined_df['label'] = combined_df['label'].replace(label_mapping)

# Optional: Filter `label_counts` and `combined_df` if designated_gene_names is not empty
if designated_gene_names:
    label_counts = label_counts[label_counts['gene_name'].isin(designated_gene_names)]
    combined_df = combined_df[combined_df['gene_name'].isin(designated_gene_names)]

# Split the data by gene_name
gyrA_labels = label_counts[label_counts['gene_name'] == 'gyrA']
rpoB_labels = label_counts[label_counts['gene_name'] == 'rpoB']
other_labels = label_counts[~label_counts['gene_name'].isin(['gyrA', 'rpoB'])]

# Sort labels
gyrA_labels = gyrA_labels.sort_values(by='freq', ascending=False)
rpoB_labels = rpoB_labels.sort_values(by='freq', ascending=False)
other_labels = other_labels.sort_values(by='freq', ascending=False)

# Combine the groups: gyrA first, rpoB second, others last
label_counts_sorted = pd.concat([gyrA_labels, rpoB_labels, other_labels], ignore_index=True)

# Create label order
label_order = label_counts_sorted['label'].tolist()

# Filter the main DataFrame
filtered_df = combined_df[combined_df['label'].isin(label_order)]

# Pivot table for heatmap
heatmap_data = filtered_df.pivot_table(index='label', columns='Pop', values='last_freq', fill_value=0)

# Reindex the heatmap data
heatmap_data = heatmap_data.reindex(manual_label_order)

# Convert the DataFrame to a numpy array for sorting
freq_mat = heatmap_data.values
timepoints = heatmap_data.columns

# Define desired cell size (in inches)
cell_width = 0.7  # Width of each cell

# Calculate the number of rows and columns
num_rows, num_columns = heatmap_data.shape

# Calculate figure size 
figsize = ((10 * cell_width), 6)

# Create a figure and axes with the calculated size
fig, ax = plt.subplots(figsize=figsize)
annot_matrix = np.where(freq_mat.T == 0, '', freq_mat.T.round(1).astype(str))

# Create the heatmap with custom settings, including inverted colors
sns.heatmap(freq_mat.T, yticklabels=timepoints, xticklabels=False, cmap=custom_map, vmin=0, vmax=1, ax=ax,
            annot=annot_matrix, fmt="", annot_kws={'size': 20, 'ha': 'center', 'va': 'center'}, cbar=False,
            linewidths=0.5,  # Thickness of the grid lines
            linecolor='lightgrey'  # Color of the grid lines
)

# Set labels and title with larger font sizes
ax.set_ylabel('Culture Number', fontsize=25)  # Set y-axis label with larger font
ax.set_xlabel('', fontsize=25)  # Set x-axis label with larger font
# ax.set_title('High Frequency Mutations of MG$^{{\\mathrm{{LEV}}}}$', fontsize=25)  # Set title with larger font

# Adjust tick label font sizes
ax.tick_params(axis='both', which='major', labelsize=15, length=0)

# Rotate x-axis labels for readability with larger font sizes
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', rotation_mode='anchor', fontsize=20)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=20)

for spine in ax.spines.values():
    spine.set_visible(True)  # Make spines visible
    spine.set_linewidth(2)   # Set the border width
    spine.set_color("black") # Set the border color

plt.subplots_adjust(bottom=0.1)
# Display the plot
plt.tight_layout()
#fig.savefig(PROJECT_PATH / "figures/final/PL_gryA+rpoB_no_labels.png", dpi=600, bbox_inches='tight')
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.colors as mcolors
plt.rcParams["font.family"] = "Nimbus Roman"
# --- Define alt_labels and manual_label_order for PL strain ---
alt_labels = {
    'gyrA G81D': 'GyrA G81D',
    'gyrA S83L': 'GyrA S83L',
    'gyrA S83W': 'GyrA S83W',
    'gyrA A119E': 'GyrA A119E',
    'gyrA D87Y': 'GyrA D87Y',
    'rpoB H1237L': 'RpoB H1237L',
    'rpoB small_indel': r'$\it{rpoB}$ indel',
    'rpoB D1064N': 'RpoB D1064N',
    'rpoB D1064G': 'RpoB D1064G',
    'rpoB E562D': 'RpoB E562D',
    'rpoB M129R': 'RpoB M129R',
    'icd H366H': 'Icd H366H',
    'rplJ P89S': 'RplJ P89S',
    'trkH L185Q': 'TrkH L185Q',
    'feaB D41E': 'FeaB D41E',
    'appY/ompT small_indel': r'$\it{appY/ompT}$ indel',
    'glvC small_indel': r'$\it{glvC}$ indel',
    'arcB G258C': 'ArcB G258C',
    'paaG S241C': 'PaaG S241C',
    'tas small_indel': r'$\it{tas}$ indel',
    'pssL small_indel': r'$\it{pssL}$ indel',
    'sucA V141A': 'SucA V141A',
    'arcA small_indel': r'$\it{arcA}$ indel',
    'ygfB P184Q': 'YgfB P184Q',
    'gltP/yjcO snp_intergenic': r'$\it{gltP/yjcO}$ snp intergenic',
    'arcA L65F': 'ArcA L65F',
    'puuA D431D': 'PuuA D431D',
    'arpA D661A': 'ArpA D661A',
    'fimB/fimE mobile_element_insertion': r'$\it{fimB/fimE}$ IS5 insertion',
    'ftp/ompC snp_intergenic': r'$\it{ftp/ompC}$ snp intergenic',
    'yhaC/rnpB snp_intergenic': r'$\it{yhaC/rnpB}$ snp intergenic',
    'yhiI V38V': 'YhiI V38V',
    'recF I355T': 'RecF I355T',
    'selB small_indel': r'$\it{selB}$ indel',
    'elaD C252C': 'ElaD C252C',
    'hlyE E42D': 'HlyE E42D',
    'yffN/yffO snp_intergenic': r'$\it{yffN/yffO}$ snp intergenic',
    'pgaC E86G': 'PgaC E86G',
}
manual_label_order = [
    # gyrA/rpoB/icd
    'gyrA G81D', 'gyrA S83L', 'gyrA S83W', 'gyrA A119E', 'gyrA D87Y',
    'rpoB H1237L', 'rpoB small_indel', 'rpoB D1064N', 'rpoB D1064G', 'rpoB E562D', 'rpoB M129R',
    'icd H366H',
    # other
    'rplJ P89S', 'trkH L185Q', 'feaB D41E',
    'appY/ompT small_indel', 'glvC small_indel',
    'arcB G258C', 'paaG S241C', 'tas small_indel',
    'pssL small_indel', 'sucA V141A',
    'arcA small_indel', 'ygfB P184Q',
    'gltP/yjcO snp_intergenic', 'arcA L65F', 'puuA D431D', 'arpA D661A',
    'fimB/fimE mobile_element_insertion', 'ftp/ompC snp_intergenic',
    'yhaC/rnpB snp_intergenic', 'yhiI V38V', 'recF I355T',
    'selB small_indel', 'elaD C252C', 'hlyE E42D',
    'yffN/yffO snp_intergenic', 'pgaC E86G'
]

# --- Use Reds colormap ---
standard_map = plt.cm.get_cmap('Reds')
new_colors = standard_map(np.linspace(0, 1, 256))
new_colors[0] = np.array([1, 1, 1, 1])  # 0 = white
custom_map = mcolors.ListedColormap(new_colors)

# --- Add 'Pop' column if missing ---
for idx, df in enumerate(af):
    if 'Pop' not in df.columns:
        df['Pop'] = idx + 1

# --- Build freq_map ---
freq_map = {
    (row['label'], row['Pop']): row['freq'][-1] if isinstance(row['freq'], (list, np.ndarray)) else row['freq']
    for df in af for _, row in df.iterrows()
}

# --- Add 'last_freq' ---
for df in af:
    df['last_freq'] = df.apply(lambda row: freq_map.get((row['label'], row['Pop']), 0), axis=1)

# --- Combine all data ---
all_data = pd.concat(af, ignore_index=True)
filtered_data = all_data[all_data['label'].isin(manual_label_order)].copy()

# --- Pivot table ---
heatmap_data = filtered_data.pivot_table(index='label', columns='Pop', values='last_freq', fill_value=0)

# --- Ensure all cultures present ---
all_cultures = range(1, 11)
heatmap_data = heatmap_data.reindex(columns=all_cultures, fill_value=0)

# --- Order rows: manual first, then rest ---
present_labels = heatmap_data.index.tolist()
present_manual_labels = [label for label in manual_label_order if label in present_labels]
remaining_labels = [label for label in present_labels if label not in manual_label_order]
final_row_order = present_manual_labels + remaining_labels
heatmap_data = heatmap_data.reindex(index=final_row_order)

# --- Plot matrix ---
freq_mat = heatmap_data.values
timepoints = heatmap_data.columns
plot_labels = [alt_labels.get(label, label) for label in heatmap_data.index]
annot_matrix = np.where(freq_mat.T == 0, '', freq_mat.T.round(1).astype(str))

# --- Plotting ---
cell_width = 0.7
figsize = ((heatmap_data.shape[0] * cell_width), 7)

fig, ax = plt.subplots(figsize=figsize)
sns.heatmap(
    freq_mat.T,
    yticklabels=timepoints,
    xticklabels=plot_labels,
    cmap=custom_map,
    vmin=0,
    vmax=1,
    ax=ax,
    annot=annot_matrix,
    fmt="",
    annot_kws={'size': 20, 'ha': 'center', 'va': 'center'},
    cbar=False,
    linewidths=0.5,
    linecolor='lightgrey'
)

# --- Style ---
ax.set_ylabel('Culture Number', fontsize=25)
ax.set_xlabel('', fontsize=25)
ax.tick_params(axis='both', which='major', labelsize=15, length=0)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', rotation_mode='anchor', fontsize=20)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=20)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(2)
    spine.set_color("black")

plt.subplots_adjust(bottom=0.1)
plt.tight_layout()

# --- Save the figure ---
fig.savefig(PROJECT_PATH / "figures/final/PL_full.png", dpi=600, bbox_inches='tight')
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.colors as mcolors
plt.rcParams["font.family"] = "Nimbus Roman"
# --- Define alt_labels and manual_label_order for PL strain ---
alt_labels = {
    'gyrA G81D': 'GyrA G81D',
    'gyrA S83L': 'GyrA S83L',
    'gyrA S83W': 'GyrA S83W',
    'gyrA A119E': 'GyrA A119E',
    'gyrA D87Y': 'GyrA D87Y',
    'rpoB H1237L': 'RpoB H1237L',
    'rpoB small_indel': r'$\it{rpoB}$ indel',
    'rpoB D1064N': 'RpoB D1064N',
    'rpoB D1064G': 'RpoB D1064G',
    'rpoB E562D': 'RpoB E562D',
    'rpoB M129R': 'RpoB M129R',

}
manual_label_order = [
    # gyrA/rpoB/icd
    'gyrA G81D', 'gyrA S83L', 'gyrA S83W', 'gyrA A119E', 'gyrA D87Y',
    'rpoB H1237L', 'rpoB small_indel', 'rpoB D1064N', 'rpoB D1064G', 'rpoB E562D', 'rpoB M129R',

]

# --- Use Reds colormap ---
standard_map = plt.cm.get_cmap('Reds')
new_colors = standard_map(np.linspace(0, 1, 256))
new_colors[0] = np.array([1, 1, 1, 1])  # 0 = white
custom_map = mcolors.ListedColormap(new_colors)

# --- Add 'Pop' column if missing ---
for idx, df in enumerate(af):
    if 'Pop' not in df.columns:
        df['Pop'] = idx + 1

# --- Build freq_map ---
freq_map = {
    (row['label'], row['Pop']): row['freq'][-1] if isinstance(row['freq'], (list, np.ndarray)) else row['freq']
    for df in af for _, row in df.iterrows()
}

# --- Add 'last_freq' ---
for df in af:
    df['last_freq'] = df.apply(lambda row: freq_map.get((row['label'], row['Pop']), 0), axis=1)

# --- Combine all data ---
all_data = pd.concat(af, ignore_index=True)
filtered_data = all_data[all_data['label'].isin(manual_label_order)].copy()

# --- Pivot table ---
heatmap_data = filtered_data.pivot_table(index='label', columns='Pop', values='last_freq', fill_value=0)

# --- Ensure all cultures present ---
all_cultures = range(1, 11)
heatmap_data = heatmap_data.reindex(columns=all_cultures, fill_value=0)

# --- Order rows: manual first, then rest ---
present_labels = heatmap_data.index.tolist()
present_manual_labels = [label for label in manual_label_order if label in present_labels]
remaining_labels = [label for label in present_labels if label not in manual_label_order]
final_row_order = present_manual_labels + remaining_labels
heatmap_data = heatmap_data.reindex(index=final_row_order)

# --- Plot matrix ---
freq_mat = heatmap_data.values
timepoints = heatmap_data.columns
plot_labels = [alt_labels.get(label, label) for label in heatmap_data.index]
annot_matrix = np.where(freq_mat.T == 0, '', freq_mat.T.round(1).astype(str))

# --- Plotting ---
cell_width = 0.7
figsize = ((heatmap_data.shape[0] * cell_width), 7)

fig, ax = plt.subplots(figsize=figsize)
sns.heatmap(
    freq_mat.T,
    yticklabels=timepoints,
    xticklabels=plot_labels,
    cmap=custom_map,
    vmin=0,
    vmax=1,
    ax=ax,
    annot=annot_matrix,
    fmt="",
    annot_kws={'size': 20, 'ha': 'center', 'va': 'center'},
    cbar=False,
    linewidths=0.5,
    linecolor='lightgrey'
)

# --- Style ---
ax.set_ylabel('Culture Number', fontsize=25)
ax.set_xlabel('', fontsize=25)
ax.tick_params(axis='both', which='major', labelsize=15, length=0)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', rotation_mode='anchor', fontsize=20)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=20)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(2)
    spine.set_color("black")

plt.subplots_adjust(bottom=0.1)
plt.tight_layout()

# --- Save the figure ---
fig.savefig(PROJECT_PATH / "figures/final/PL_sub_full.png", dpi=600, bbox_inches='tight')
plt.show()
